# 🤖 Classification de CVs — Prédiction de `passed_next_stage`

Ce notebook entraîne plusieurs modèles de classification pour prédire si un CV passe à l'étape suivante du recrutement.

**Pipeline :**
1. Chargement & aperçu des données
2. Définition des features (numériques, catégorielles, texte)
3. Split train / test stratifié
4. Pré-traitement (StandardScaler + OneHotEncoder + TF-IDF)
5. Entraînement de 4 modèles avec SMOTE (gestion du déséquilibre)
6. Évaluation (ROC-AUC, F1, matrices de confusion)
7. Importance des features (meilleur modèle)

## 1. Imports

In [ ]:
import warnings

from matplotlib.pyplot import grid

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Versions
import sklearn, imblearn
print(f'pandas     {pd.__version__}')
print(f'numpy      {np.__version__}')
print(f'sklearn    {sklearn.__version__}')
print(f'imblearn   {imblearn.__version__}')
print(f'seaborn    {sns.__version__}')

## 2. Chargement & aperçu des données

In [ ]:
df = pd.read_csv('../data/cv_dataset.csv')   # adapter le chemin si nécessaire

print(f'Shape : {df.shape}')
df.head()

In [ ]:
print('Types de colonnes :')
print(df.dtypes)
print()
print('Valeurs manquantes :')
print(df.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution de la cible
counts = df['passed_next_stage'].value_counts()
axes[0].bar(['Refusé (0)', 'Sélectionné (1)'], counts.values,
            color=['#e74c3c', '#2ecc71'], alpha=0.85, edgecolor='white')
axes[0].set_title('Distribution de la variable cible')
axes[0].set_ylabel('Nombre de CVs')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Taux de sélection par rôle
role_rate = df.groupby('target_role')['passed_next_stage'].mean().sort_values()
role_rate.plot(kind='barh', ax=axes[1], color='steelblue', alpha=0.8)
axes[1].set_title('Taux de sélection par rôle')
axes[1].set_xlabel('Taux de sélection')
axes[1].axvline(df['passed_next_stage'].mean(), color='red', linestyle='--',
                label=f"Moyenne globale ({df['passed_next_stage'].mean():.1%})")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"Taux de sélection global : {df['passed_next_stage'].mean():.1%}")

## 3. Définition des features

In [ ]:
TARGET = 'passed_next_stage'
DROP_COLS = ['cv_id']

NUMERIC_FEATURES = [
    'age',
    'distance_ville_haute_km',
    'total_experience_years',
    'total_gap_months',
    'nb_gaps',
    'education_score',
    'number_of_experiences',
    'lang_fr',
    'lang_en',
    'lang_de',
    'lang_es',
    'lang_it',
    'lang_other_score_sum',
]

CATEGORICAL_FEATURES = [
    'target_role',
    'education_degree',
    'education_field',
    'education_school',
]

TEXT_SKILLS         = 'skills'
TEXT_CERTIFICATIONS = 'certifications'

X = df.drop(columns=DROP_COLS + [TARGET])
y = df[TARGET]

print(f'Features numériques    : {len(NUMERIC_FEATURES)}')
print(f'Features catégorielles : {len(CATEGORICAL_FEATURES)}')
print(f'Colonnes texte         : 2 (skills + certifications)')
print(f'Shape de X             : {X.shape}')
print(f'Shape de y             : {y.shape}')

## 4. Split train / test stratifié

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {X_train.shape[0]} lignes  —  '
      f"Sélectionnés : {y_train.sum()} ({y_train.mean():.1%})")
print(f'Test  : {X_test.shape[0]} lignes  —  '
      f"Sélectionnés : {y_test.sum()} ({y_test.mean():.1%})")

## 5. Construction du pré-traitement

| Colonne | Transformation |
|---------|----------------|
| Numériques | `StandardScaler` |
| Catégorielles | `OneHotEncoder(handle_unknown='ignore')` |
| `skills` | `TfidfVectorizer(max_features=50, ngram_range=(1,2))` |
| `certifications` | `TfidfVectorizer(max_features=30)` |

In [ ]:
numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

skills_transformer = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50, ngram_range=(1, 2)))
])

certif_transformer = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=30, ngram_range=(1, 1)))
])

preprocessor = ColumnTransformer(transformers=[
    ('num',    numeric_transformer,     NUMERIC_FEATURES),
    ('cat',    categorical_transformer, CATEGORICAL_FEATURES),
    ('skills', skills_transformer,      TEXT_SKILLS),
    ('certif', certif_transformer,      TEXT_CERTIFICATIONS),
])

print('Preprocessor créé ✓')

## 6. Définition des modèles

Chaque pipeline inclut **SMOTE** pour rééquilibrer les classes avant l'entraînement.

In [ ]:
models = {
    'Logistic Regression': ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote',        SMOTE(random_state=42)),
        ('classifier',   LogisticRegression(
            max_iter=1000, random_state=42, class_weight='balanced'
        )),
    ]),
    'Random Forest': ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote',        SMOTE(random_state=42)),
        ('classifier',   RandomForestClassifier(
            n_estimators=200, random_state=42, class_weight='balanced'
        )),
    ]),
    'Gradient Boosting': ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote',        SMOTE(random_state=42)),
        ('classifier',   GradientBoostingClassifier(
            n_estimators=200, random_state=42
        )),
    ]),
    'SVM (RBF)': ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote',        SMOTE(random_state=42)),
        ('classifier',   SVC(
            probability=True, random_state=42, class_weight='balanced'
        )),
    ]),
}

print(f'{len(models)} modèles définis :', list(models.keys()))

## 7. Entraînement & évaluation

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, pipeline in models.items():
    print(f'\n{'─'*45}')
    print(f'  {name}')
    print(f'{'─'*45}')

    # Cross-validation sur le train
    cv_scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=cv_strategy, scoring='f1'
    )

    # Entraînement final
    pipeline.fit(X_train, y_train)

    # Prédictions sur le test
    y_pred  = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_proba)

    results[name] = {
        'pipeline':     pipeline,
        'y_pred':       y_pred,
        'y_proba':      y_proba,
        'cv_f1_mean':   cv_scores.mean(),
        'cv_f1_std':    cv_scores.std(),
        'roc_auc':      roc_auc,
    }

    print(f'  CV F1 (5-fold) : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
    print(f'  ROC-AUC (test) : {roc_auc:.3f}')
    print(classification_report(
        y_test, y_pred,
        target_names=['Refusé', 'Sélectionné']
    ))

## 8. Synthèse des scores

In [ ]:
summary = pd.DataFrame({
    'Modèle':       list(results.keys()),
    'ROC-AUC':      [r['roc_auc']    for r in results.values()],
    'CV F1 (mean)': [r['cv_f1_mean'] for r in results.values()],
    'CV F1 (std)':  [r['cv_f1_std']  for r in results.values()],
}).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

best_name = summary.iloc[0]['Modèle']
print(f'🏆 Meilleur modèle : {best_name}')
summary

## 9. Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Évaluation des modèles de classification de CVs',
             fontsize=16, fontweight='bold')

# ── 9a. Comparaison ROC-AUC vs CV F1 ──
ax = axes[0, 0]
model_names = list(results.keys())
roc_aucs    = [results[m]['roc_auc']    for m in model_names]
cv_f1_means = [results[m]['cv_f1_mean'] for m in model_names]
x     = np.arange(len(model_names))
width = 0.35
bars1 = ax.bar(x - width/2, roc_aucs,    width,
               label='ROC-AUC (test)', color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + width/2, cv_f1_means, width,
               label='CV F1 (train)', color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
ax.set_ylim(0, 1)
ax.set_title('Comparaison des scores par modèle')
ax.set_ylabel('Score')
ax.legend()
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

# ── 9b. Courbes ROC ──
ax = axes[0, 1]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.2f})")
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_title('Courbes ROC')
ax.set_xlabel('Taux de Faux Positifs')
ax.set_ylabel('Taux de Vrais Positifs')
ax.legend(fontsize=8)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

# ── 9c. Matrice de confusion (meilleur modèle) ──
ax = axes[1, 0]
cm   = confusion_matrix(y_test, results[best_name]['y_pred'])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=['Refusé', 'Sélectionné']
)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matrice de confusion — {best_name}')

# ── 9d. Feature importance (Random Forest) ──
ax = axes[1, 1]
rf_pipe    = results['Random Forest']['pipeline']
rf_model   = rf_pipe.named_steps['classifier']
prep_fit   = rf_pipe.named_steps['preprocessor']

num_names    = NUMERIC_FEATURES
cat_names    = list(prep_fit.named_transformers_['cat']
                    .named_steps['ohe']
                    .get_feature_names_out(CATEGORICAL_FEATURES))
skills_names = [f'skill_{v}' for v in prep_fit.named_transformers_['skills']
                .named_steps['tfidf'].get_feature_names_out()]
certif_names = [f'certif_{v}' for v in prep_fit.named_transformers_['certif']
                .named_steps['tfidf'].get_feature_names_out()]

all_feature_names = num_names + cat_names + skills_names + certif_names
importances = pd.Series(rf_model.feature_importances_, index=all_feature_names)
top15 = importances.nlargest(15).sort_values()
top15.plot(kind='barh', ax=ax, color='teal', alpha=0.8)
ax.set_title('Top 15 features — Random Forest')
ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 10. Analyse du seuil de confiance

La courbe ci-dessous montre comment **Precision**, **Recall** et **F1** évoluent selon le seuil choisi.
La forme en **S** de la précision est caractéristique : elle monte rapidement puis se stabilise.

> **Comment lire ce graphique ?**
> - Seuil **bas** → recall élevé (peu de bons CVs ratés) mais plus de faux positifs
> - Seuil **haut** → sélection stricte mais risque de rater de bons profils
> - La **ligne verte** indique le seuil qui maximise le F1-score

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

best_probas = results[best_name]["y_proba"]
thresholds  = np.linspace(0.01, 0.99, 200)

precisions, recalls, f1s = [], [], []
for t in thresholds:
    preds = (best_probas >= t).astype(int)
    if preds.sum() == 0:
        precisions.append(np.nan)
        recalls.append(0.0)
        f1s.append(0.0)
    else:
        precisions.append(precision_score(y_test, preds, zero_division=0))
        recalls.append(recall_score(y_test, preds, zero_division=0))
        f1s.append(f1_score(y_test, preds, zero_division=0))

precisions = np.array(precisions)
recalls    = np.array(recalls)
f1s        = np.array(f1s)

# Seuil optimal = F1 max
optimal_idx       = np.nanargmax(f1s)
optimal_threshold = thresholds[optimal_idx]
optimal_f1        = f1s[optimal_idx]
optimal_precision = precisions[optimal_idx]
optimal_recall    = recalls[optimal_idx]

print(f"Seuil optimal (F1 max) : {optimal_threshold:.2f}")
print(f"  -> Precision : {optimal_precision:.3f}")
print(f"  -> Recall    : {optimal_recall:.3f}")
print(f"  -> F1        : {optimal_f1:.3f}")

# Graphique
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(thresholds, precisions, color="#e74c3c", linewidth=2.5, label="Precision")
ax.plot(thresholds, recalls,    color="#3498db", linewidth=2.5, label="Recall")
ax.plot(thresholds, f1s,        color="#f39c12", linewidth=2.5, label="F1-score", linestyle="--")

# Zone ombrée autour du seuil optimal
ax.axvspan(optimal_threshold - 0.05, optimal_threshold + 0.05,
           alpha=0.12, color="green", label="Zone optimale")

# Ligne seuil optimal
ax.axvline(optimal_threshold, color="#27ae60", linewidth=2, linestyle="-.",
           label=f"Seuil optimal = {optimal_threshold:.2f}")

# Ligne seuil par défaut
ax.axvline(0.5, color="grey", linewidth=1.5, linestyle=":",
           label="Seuil par défaut = 0.50")

# Annotation
ax.annotate(
    f"F1={optimal_f1:.2f}\nP={optimal_precision:.2f}  R={optimal_recall:.2f}",
    xy=(optimal_threshold, optimal_f1),
    xytext=(optimal_threshold + 0.08, optimal_f1 - 0.15),
    fontsize=10, color="#27ae60", fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="#27ae60", lw=1.5)
)

ax.set_xlabel("Seuil de confiance", fontsize=13)
ax.set_ylabel("Score", fontsize=13)
ax.set_title(
    f"Courbe Precision / Recall / F1 selon le seuil de confiance ({best_name})",
    fontsize=14, fontweight="bold"
)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Appliquer le seuil optimal pour la suite
THRESHOLD = optimal_threshold
print(f"\nSeuil retenu pour les prédictions : {THRESHOLD:.3f}")

## 11. Prédire un nouveau CV

Exemple d'utilisation du meilleur modèle sur un CV fictif.

In [ ]:
new_cv = pd.read_csv("../data/first_json/first_cv_dataset.csv")
# Récupération du meilleur modèle
best_pipe = results[best_name]['pipeline']

# Calcul des probabilités pour la classe 1 (passer à l'étape suivante)
proba = best_pipe.predict_proba(new_cv)[:, 1]
new_cv['proba_selection'] = proba

# --- Application du seuil optimal dynamique ---
new_cv['prediction'] = np.where(proba >= THRESHOLD, '✅ Sélectionné', '❌ Refusé')

print(f"Modèle utilisé : {best_name}")
print(f"Seuil appliqué : {THRESHOLD:.4f}")
print(f"Nombre de CVs évalués : {len(new_cv)}\n")

# Sélection des colonnes pertinentes pour l'affichage (si elles existent dans votre CSV)
colonnes_affichage = ['cv_id', 'target_role', 'prediction', 'proba_selection']
colonnes_affichage = [c for c in colonnes_affichage if c in new_cv.columns]

# Affichage des premiers résultats
display(new_cv[colonnes_affichage].head())